What You Can Ignore for Now

Self-RAG
Corrective RAG
Adaptive RAG
Agentic RAG
GraphRAG
Multi-Modal RAG
Temporal RAG
HyDE / Step-back / Complex Query Transformation

These are advanced and not required for most real projects.

What is RAG?

RAG = Retrieval-Augmented Generation

It is a technique that combines two things:

Retrieval — Searching and pulling relevant information from an external knowledge source (documents, databases, websites, etc.)

Generation — Using a Large Language Model (LLM) to generate an answer based on that retrieved information

User Question
      ↓
Retrieve relevant chunks from your knowledge base
      ↓
Put those chunks + the question into the LLM prompt
      ↓
LLM generates an answer grounded in the retrieved data

Instead of asking the LLM to answer only from its internal training data, you give it fresh, relevant context at query time.

Why Does RAG Exist?

LLMs only know data up to their training date and LLM cannot access your company’s internal docs, emails, code, etc. and General models are weak in specialized domains (legal, medical, finance) so You can use RAG to feed latest documents and data with Inject domain-specific documents.

Think of an LLM as a very smart student who has read a lot of books until a certain year.

Without RAG → The student answers only from memory what they read.
With RAG → Before answering, the student is allowed to quickly look up the relevant pages from a library (your documents) and then answers using those pages.

In RAG, the source (also called knowledge base / document store / corpus) is the collection of original data that you want the system to retrieve from.

Type of Source may be PDFs, Word, Markdown, TXT, HTMLFilesWeb / OnlineWebsite pages, blogs, documentation sitesCrawled pagesDatabasesSQL tables, Notion, Confluence, SharePointStructured + text, CodeGitHub repos, internal codebasesSource filesConversations / TicketsSlack, Zendesk, emails, support ticketsChat logsMulti-modalImages + captions, tables, slidesMixed

These raw sources are not directly given to the LLM.
They go through this process:
Raw Source Documents
        ↓
Cleaning + Chunking
        ↓
Embedding (convert text → vectors)
        ↓
Store in Vector Database (or local index)

What is Indexing?

Indexing is the process of converting your raw source documents into a searchable format that the RAG system can quickly retrieve from.

Why Indexing is Needed

LLMs cannot efficiently search through thousands of pages every time a user asks a question.
Indexing pre-processes the data so that retrieval becomes fast and accurate.
Without good indexing → poor retrieval → bad answers (even with a strong LLM).

Indexing Pipeline (Step-by-Step)

Here is the standard indexing flow:

(1)Load Documents

Read PDFs, Markdown, HTML, Word files, database records, etc.

(2)Clean the Data

Remove headers, footers, ads, duplicate content, broken text, etc.

(3)Chunking

Split long documents into smaller pieces (chunks).
This is one of the most important decisions in RAG.

(4)Add Metadata (optional but powerful)

Attach useful information to each chunk:
Source file name
Page number
Author
Date
Section / category
Document type

(5)Embedding

Convert each chunk into a numerical vector using an embedding model.

(6)Store in Vector Database / Index (knowledge based)

Save the vectors + original text + metadata so they can be searched later.

What is Retrieval?

Retrieval is the step where the system finds the most relevant pieces of information from your indexed knowledge base based on the user’s question.

How Retrieval Works (Simple View)

User asks a question

The question is converted into a vector (using the same embedding model used during indexing)

The system searches the vector index for the most similar chunks

Top-k most relevant chunks are returned

This is called Dense Retrieval (semantic search).

What is Augment?

Augment is the step where the retrieved chunks are combined with the user’s original question to create a rich prompt that is sent to the LLM.The goal is to give the LLM only the necessary context so it can generate an accurate, grounded answer.

User Question + Retrieved Chunks
              ↓
           Augment
              ↓
        Final Prompt
              ↓
           Generate

What is Generate?

Generate is the final step of the RAG pipeline, where the LLM produces the answer using the augmented prompt (user question + retrieved context).This is the stage where the language model actually writes the response.

1. Indexing     → Prepare searchable knowledge
2. Retrieve     → Find relevant chunks
3. Augment      → Build the prompt with context
4. Generate     → LLM produces the final answer

What is a Hallucination?

When an AI makes up information that is not true, we call it a hallucination.

Example:
Question: Who is the CEO of Tesla in 2026?

AI (hallucinating): "The CEO of Tesla is John Smith."

→ This is wrong.CEo is elon musk. The AI invented the name.

The AI sounds confident, but the information is fake.

Why does it happen?

AI models are not databases.
They don’t actually “know” facts.
They just predict what words should come next based on patterns they learned.
When they don’t know something, they often still try to give an answer — and that answer can be wrong.

How RAG helps

Without RAG:

AI answers only from its memory → high chance of making things up.

With RAG:

We give the AI real documents (your data) and tell it:
“Only use this information to answer.”
This greatly reduces hallucinations because the AI has real sources to look at.RAG reduces hallucinations a lot, but it does not remove them 100%.

What is Knowledge Cutoff?

Every Large Language Model (LLM) is trained on data only up to a certain date.
After that date, the model has no knowledge of new events, facts, or information.
This date is called the Knowledge Cutoff.

Example:

Suppose an LLM was trained with data until December 2024.
If you ask it in 2026: “Who won the 2025 Cricket World Cup?”
The model does not know the answer because that information did not exist during its training.The model can only answer based on what it learned during training. Anything after the cutoff date is invisible to it.

How RAG Solves the Knowledge Cutoff Problem

RAG allows you to give the model fresh information at the time of the question.
Instead of relying only on the model’s old training data, you:

Store the latest documents in a knowledge base
Retrieve the relevant ones when a question comes
Give those documents to the model
Ask the model to answer using that new information

Result: The model can answer questions about information that was created after its training cutoff.

Chunking Strategies in RAG

Chunking means splitting large documents into smaller pieces (chunks) before creating embeddings.
Good chunking = better retrieval. Bad chunking = poor answers.

Here are the main strategies:

(1)What is Fixed-Size Chunking?

This is the simplest way to split documents.
You break the text into pieces of equal size, regardless of sentences or paragraphs.
Example:

Chunk size = 500 tokens
Document has 1800 tokens

Chunk 1 → tokens 1 to 500
Chunk 2 → tokens 501 to 1000
Chunk 3 → tokens 1001 to 1500
Chunk 4 → tokens 1501 to 1800

Pure fixed-size often cuts important sentences in half.
To reduce this problem, we add overlap.
Example:

Chunk size = 500 tokens
Overlap = 50 tokens
Chunk 1: tokens 1 – 500
Chunk 2: tokens 451 – 950
Chunk 3: tokens 901 – 1400
The overlapping part helps the model keep some context from the previous chunk.

(2)What is Recursive Character Chunking?

This is one of the most popular and practical chunking strategies used in real RAG systems.
Instead of cutting the text at a fixed number of tokens, it tries to split the document at natural boundaries.
It follows a priority order of separators:

Double new line (\n\n) → Paragraphs
Single new line (\n) → Lines
Sentences (. , ? , ! )
Words (space)
Characters (last resort)

It keeps trying smaller separators only when the chunk is still too big.
How it Works (Simple Example)
Suppose you set:

Maximum chunk size = 500 tokens
Separators = ["\n\n", "\n", ". ", " ", ""]

Process:

First try to split by paragraphs (\n\n)
If any paragraph is still larger than 500 tokens → split that paragraph by lines (\n)
If still too big → split by sentences
If still too big → split by words
Only if necessary → split by characters

This way, it tries hard to keep meaningful pieces of text together.

(3)What is Semantic Chunking?

Semantic Chunking splits the document based on meaning, not just size or structure.
It tries to keep sentences that talk about the same topic together, and creates a new chunk only when the topic changes significantly.Simple idea:
“If the meaning stays similar → keep in same chunk.
If the meaning changes a lot → start a new chunk.”
Example:-
The company was founded in 2015 in Bangalore. 
It started with 5 employees and focused on AI tools.
In 2022, the company raised Series B funding.
The funding was used to expand the engineering team.
The weather in Bangalore is pleasant in December.

Semantic Chunking result (approximate):

Chunk 1: Company founding + early stage + funding + team expansion
Chunk 2: Weather in Bangalore

Even though the weather sentence is close in position, its meaning is different, so it becomes a separate chunk.

(4)What is Late Chunking?

Late Chunking is an advanced technique where you delay the splitting of the document until after you have created embeddings for the full text (or large parts of it).
In normal chunking:

First split the document into chunks
Then create embeddings for each chunk

In Late Chunking:

First create embeddings while considering the full document context
Then decide where to split (create chunks)

This helps each chunk keep richer contextual meaning.

Why Late Chunking Exists

Normal chunking has a big problem:
When you split text early, each chunk is embedded in isolation.
The embedding model cannot see the surrounding context, so the meaning of some sentences can become weaker or unclear.
Late Chunking tries to fix this by letting the model see more context before finalizing the chunks.

USECASE:-

Just learning RAG ------ Fixed-size + OverlapMost 
normal projects --------- Recursive
Need better quality than recursive--------Semantic Chunking
Maximum quality and advanced system-------Late Chunking

What are Embedding Models?

An embedding model converts text into a list of numbers (a vector) that captures the meaning of the text.

Example:

Sentence: “I love machine learning”
Output: [0.12, -0.45, 0.88, ..., 0.03] (e.g. 384 or 1024 numbers)
These vectors allow the system to find similar meanings, even if the exact words are different.

Good embedding model → finds relevant chunks even with different wording
Weak embedding model → misses relevant information or brings noise

Chunking decides what pieces you have.
Embedding model decides how well those pieces can be found.

Always use the same embedding model for indexing and querying.

What is a Vector Representation?

A vector representation (also called an embedding) is a way of turning text into a list of numbers that captures its meaning.

xample:

Text: “The cat is sleeping”
Vector: [0.23, -0.41, 0.87, 0.12, ..., 0.05]

This list of numbers is the vector representation of that sentence.

Why Do We Use Vectors?

Computers cannot directly understand meaning like humans do.
But they are very good at working with numbers.
By converting text into vectors:

We can measure how similar two pieces of text are
We can search for meaning instead of just exact words
We can find relevant documents even if the wording is different
Even though the words are different, the vectors will be close if the meaning is similar.Ex- “I love dogs” is similarity to “I really like dogs”.

How Vectors Work in RAG

During Indexing:
Every chunk is converted into a vector
These vectors are stored in a vector database

During Retrieval:
The user question is also converted into a vector
The system finds chunks whose vectors are closest to the question vector


This is called semantic search.
Vector representations allow RAG to search by meaning instead of exact keywords.
This is the core reason RAG can find relevant information even when the user uses different words.

What are Similarity Metrics?

Similarity metrics measure how close two vectors are or It is a way to measure how similar two pieces of text are (after they are converted into vectors).Think of it like measuring distance between two points.
The closer the vectors → the more similar the meaning.
Imagine you have many people standing in a big field.

Let People who like similar things stand close to each other.
People who like different things stand far apart.

When a new person comes (the user question), we look for the people standing closest to them.
Similarity metric is the method we use to decide “who is closest”.

(1)What is Cosine Similarity?
Cosine Similarity measures how similar two vectors are by looking at the angle between them.

If the angle is small → High similarity
If the angle is large → Low similarity

It does not care about the length of the vectors, only the direction.
Imagine two arrows:

If both arrows point almost the same way → Cosine Similarity is close to 1 (very similar)
If they point in completely different directions → Cosine Similarity is close to 0 (not similar)
If they point in opposite directions → Cosine Similarity is close to -1

Question: “What is machine learning?”
Chunk A: “Machine learning is a type of artificial intelligence...”
→ Cosine Similarity = 0.89 (Very high → highly relevant)
Chunk B: “The weather in Mumbai is hot today...”
→ Cosine Similarity = 0.12 (Very low → not relevant)
The system will pick Chunk A because its score is much higher.
Cosine Similarity is used at max place

(2)What is Dot Product similarity?

Dot Product is another way to measure how similar two vectors are.
It multiplies the corresponding numbers in the two vectors and adds them up.
Simple idea:

If two vectors point in a similar direction → Dot Product is high
If they point in different directions → Dot Product is low

Think of two arrows again:

If both arrows are pointing roughly the same way and are long → Dot Product becomes large
If they point in different directions → Dot Product becomes small or negative

Ex-
Let’s take two small example vectors:

Vector A (Question): [1, 2, 3]
Vector B (Chunk): [4, 5, 6]

Multiply the numbers in the same position and add them:
(1 × 4) + (2 × 5) + (3 × 6)
= 4 + 10 + 18
= 32 Higher number = more similar.

How Cosine Similarity is Calculated
Cosine Similarity uses the Dot Product, but also divides by the lengths of both vectors.
Step 1: Calculate Dot Product → 32
Step 2: Calculate length of Vector A
√(1² + 2² + 3²) = √(1+4+9) = √14 ≈ 3.74
Step 3: Calculate length of Vector B
√(4² + 5² + 6²) = √(16+25+36) = √77 ≈ 8.77
Step 4: Divide
Cosine Similarity = 32 / (3.74 × 8.77) ≈ 32 / 32.8 ≈ 0.98
Result ≈ 0.98 (very similar)

(3)What is Euclidean Distance similarity?

Euclidean Distance measures the straight-line distance between two vectors.
It answers the question:
“How far apart are these two points?”

0 = Exactly the same
Larger number = More different

This is a distance metric (lower is better), unlike Cosine Similarity (higher is better).
Imagine two points on a piece of paper:

If the points are on top of each other → Distance = 0
If the points are far apart → Distance is large

Euclidean Distance calculates that straight-line gap.
How it is Calculated (Simple Example)
Take the same vectors:

Vector A: [1, 2, 3]
Vector B: [4, 5, 6]

Formula:
√[ (1-4)² + (2-5)² + (3-6)² ]
= √[ (-3)² + (-3)² + (-3)² ]
= √[ 9 + 9 + 9 ]
= √27
≈ 5.20

Always start with Cosine Similarity.
Only change it if you have a clear reason (normalized vectors + speed needs, or model requirement).

What is Approximate Nearest Neighbor Search (ANN)?
In RAG, after converting the question into a vector, we need to find the most similar vectors (chunks) from the vector database.
Exact search = Check every single vector (very accurate but very slow).
Approximate Nearest Neighbor (ANN) = Find vectors that are almost the nearest ones, but much faster.It gives a small trade-off in accuracy for a huge gain in speed.In real RAG systems (especially with large data), we almost always use ANN.In practice, it give 95–99% accuracy is more than enough for RAG, and the speed benefit is massive.

What is HNSW?

HNSW is currently the most popular and effective algorithm for Approximate Nearest Neighbor (ANN) search.
It is used by most modern vector databases (Qdrant, Weaviate, Milvus, Chroma, FAISS, etc.).

What is IVF?

IVF stands for Inverted File Index.
It is another popular Approximate Nearest Neighbor (ANN) method used in vector search.
It works by dividing all vectors into groups (clusters) and only searching inside the most relevant groups.

Simple Ex-
Imagine you have thousands of books in a big library:
Instead of checking every book, you first divide them into categories (clusters)
When someone asks a question, you only search inside the 2–3 most relevant categories
This is exactly how IVF works.

What is Prompt Construction in RAG?

After retrieving the relevant chunks, we need to combine them with the user question and clear instructions before sending everything to the LLM.
This step is called prompt construction (part of the Augment stage).
A good prompt tells the LLM:

What role it should play
That it must use only the given context
How to format the answer
What to do if the answer is not in the context

Here is the most common and effective structure:

You are a helpful assistant. Answer the question based only on the context provided below. 
If the answer is not in the context, say "I don't know based on the available information."

Context:
{retrieved_chunk_1}
{retrieved_chunk_2}
{retrieved_chunk_3}

Question: {user_question}

Answer:

Why This Structure Works

Role instruction ----"Sets the behavior of the LLM“
Use only the context"-------Reduces hallucination
“I don’t know” ------- instructionPrevents making up answers
Context section--------Provides the retrieved knowledge
Question-------Clear user query
“Answer:”------Guides the model to start responding

Improved Basic Prompt (Recommended)

You are a knowledgeable assistant. Your task is to answer the user's question using only the information provided in the Context section below.

Rules:
- Only use information from the Context.
- If the Context does not contain enough information, reply with "I don't know based on the available information."
- Be concise and clear.
- Do not make up facts.

Context:
{retrieved_chunks}

Question: {user_question}

Answer:

Version with Citations (Better)

You are a helpful assistant. Answer the question using only the provided context. 
Cite the sources using [1], [2], etc.

Context:
[1] {chunk_1}
[2] {chunk_2}
[3] {chunk_3}

Question: {user_question}

Answer:

This version makes the answer more trustworthy.

What is Sparse Search?

Sparse Search means searching based on exact words (keywords), not meaning.
It is called “sparse” because it uses word-based representations (most values are zero), unlike dense embeddings which are full of numbers.

User Query → Match exact keywords → Rank using BM25 → Return results

PostgreSQL supports sparse search using Full Text Search (tsvector + tsquery).
SQL

When to Use Sparse Search

Searching for exact IDs, codes, names
Technical documents with specific terms
When users type exact keywords

What is Dense Search?

Dense Search means searching using embeddings (vectors).
It is called “dense” because the vector contains dense numerical information about the meaning of the text (not just keywords).

Text → Embedding Model → Dense Vector → Similarity Search

How Dense Search Works

Convert document chunks into embeddings → Store in database
Convert user question into embedding
Find vectors that are closest to the question vector
Return the most similar chunks

This is Dense Search.

SELECT content, source, page_number
FROM documents
ORDER BY embedding <=> '[question_embedding]'
LIMIT 5;

What is Hybrid Search?

Hybrid Search combines two different search methods deense and sparse search.By combining both, you get better results than using only one method.

User Query
     ↓
┌────────────────────┬────────────────────┐
│  Dense Search      │  Sparse Search     │
│  (Vector/Embedding)│  (BM25 / Keyword)  │
└────────────────────┴────────────────────┘
     ↓                        ↓
   Top results             Top results
     └──────────┬───────────┘
                ↓
        Score Fusion (RRF or Weighted)
                ↓
          Final Top K Results

What is RRF?

Reciprocal Rank Fusion (RRF) is a simple method to combine results from multiple search systems into one final ranked list.
It is most commonly used in Hybrid Search to merge:

Dense Search (Vector) results
Sparse Search (BM25) results

Formula:-
Score = 1 / (k + rank)

Where:

rank = position of the document in the list (starts from 1)
k = constant (usually 60)
For every document, calculate this score from each search method and add them.

Let’s understand Reciprocal Rank Fusion with a real situation.
Situation
User asks:
“How many casual leaves do employees get?”
We run two searches:

Step 1: Dense Search (Vector Search) Results

Rank    Document        Why it came
1        Doc A      Talks about leave policy (meaning match)
2,       Doc B       Mentions casual leave
3        Doc C       General HR policy

Step 2: Sparse Search (Keyword Search) Results

Rank   Document    Why it came
1       Doc B    Contains exact words “casual leave”
2       Doc D    Contains “casual leave days”
3       Doc A    Contains “leave”

Step 3: Apply RRF Formula

Formula:
textScore = 1 / (60 + rank)

Now calculate for each document:

Doc A:

Dense Rank = 1 → Score = 1/(60+1) = 0.01639
Sparse Rank = 3 → Score = 1/(60+3) = 0.01587
Total = 0.03226

Doc B:

Dense Rank = 2 → Score = 1/(60+2) = 0.01613
Sparse Rank = 1 → Score = 1/(60+1) = 0.01639
Total = 0.03252

Doc C:

Dense Rank = 3 → Score = 1/(60+3) = 0.01587
Sparse Rank = None → Score = 0
Total = 0.01587

Doc D:

Dense Rank = None → Score = 0
Sparse Rank = 2 → Score = 1/(60+2) = 0.01613
Total = 0.01613


FinalRank   Document   Final Score       Reason
1            Doc B      0.03252     Strong in both Vector + Keyword
2            Doc A      0.03226     Strong in Vector, okay in Keyword"
3            Doc D      0.01613     Only strong in Keyword
4            Doc C      0.01587     Only appeared in Vector search


What Happened?

Doc B became Rank 1 because it performed well in both searches.

What is Reranking?

Imagine this:
You asked a question, and the system found 20 possible answers.
But not all of them are equally good.
Reranking means:
→ Re-checking those 20 answers carefully and picking the best 3–5.

What is a Cross-Encoder?

A Cross-Encoder is a type of model used mainly for reranking.
It takes the question and the document together as input and directly tells how relevant they are.

Simple Example
Question:
“How many casual leaves do employees get?”
Document:
“Employees are entitled to 12 days of casual leave every year.”

Cross-Encoder process:

Combines both texts together
Understands the relationship between them
Gives a relevance score (e.g., 0.92)

Why Cross-Encoders are Accurate

Because they can see both the question and the document at the same time, they understand the relationship much better than normal embedding models.

Popular Cross-Encoder Models

    Model                  Type            Notes
BAAI/bge-reranker-base   Cross-Encoder    Strong & popular
BAAI/bge-reranker-large  Cross-Encoder    Higher accuracy
BAAI/bge-reranker-v2-m3  Cross-Encoder    Multilingual
ms-marco-MiniLM          Cross-Encoder    Older but still used

Typical Usage Flow

1. Vector Search → Get top 30 chunks (fast)
2. Cross-Encoder → Rerank those 30 chunks (accurate)
3. Take top 5
4. Send to LLM


What is Cohere Reranker?

Cohere provides a managed Reranking API that helps improve search results in RAG systems.
Instead of running a cross-encoder yourself, you send the query + documents to Cohere, and it returns them in order of relevance.

Query + List of Documents → Cohere Rerank API → Sorted documents by relevance

1. Do Vector / Hybrid Search → Get top 20–50 documents
2. Send them to Cohere Reranker
3. Cohere re-orders them by relevance
4. Take top 3–5 results
5. Send to LLM


What is bge-reranker?

bge-reranker is a family of open-source reranker models created by BAAI (Beijing Academy of Artificial Intelligence).
It is a Cross-Encoder model used to re-order search results by relevance.It is open source and free.No need to send data to external API

When to Use bge-reranker

You want good accuracy without paying for APIs
You prefer open-source stack
You can run models locally or on your server
Privacy is important (data stays with you)

What is an LLM Reranker?
An LLM Reranker uses a Large Language Model (like GPT, Claude, Llama, etc.) to decide which documents are most relevant to the question.
Instead of using a specialized reranker model (like bge-reranker or Cohere), you ask the LLM itself to rank the documents.


bge-reranker (v2-m3 family) is mostly used due to open source in productions.After that Cohere Rerank


What is Query Transformation?

Query Transformation means changing or expanding the user’s original question before searching, so that retrieval becomes better.

Original User Question
        ↓
Query Transformation
        ↓
Improved / Multiple Queries
        ↓
Vector Search / Hybrid Search

A poorly written question often leads to poor retrieval.Query transformation fixes that.
Why Do We Need It?
Users often ask questions that are:

Too vague,Too short
Missing important keywords
Written in a different style from the documents

Query transformation improves the chance of finding the right chunks.

Main Query Transformation Techniques:-

(1)What is Multi-Query Transformation?

Multi-Query Transformation means generating multiple different versions of the user’s question and then searching with all of them.
Original Question
       ↓
LLM generates 3–5 different versions
       ↓
Search with all versions
       ↓
Combine the results (usually with RRF)

This increases the chance of finding the right information.
Original User Question:
“How many casual leaves do employees get?”
Generated Multi-Queries:

“What is the casual leave policy for employees?”
“Number of casual leave days allowed per year”
“How much casual leave can staff take annually?”
“Casual leave entitlement in the company policy”

Now the system searches using all these queries and combines the results.

Sample Prompt to Generate Multi-Queries:-

You are a helpful assistant that generates multiple search queries.

Generate 4 different versions of the following question for document retrieval.
Keep them clear and diverse.

Original Question: {user_question}

Queries:


(2)What is HyDE?

HyDE stands for Hypothetical Document Embeddings.
Instead of searching with the user’s question directly, we:

Ask an LLM to generate a hypothetical answer
Convert that hypothetical answer into an embedding
Use that embedding to search in the vector database

User Question
     ↓
LLM generates a Hypothetical Answer
     ↓
Convert Hypothetical Answer → Embedding
     ↓
Search with this embedding

Why Does HyDE Work?
User questions and actual documents are often written in different styles.

Questions are usually short and interrogative
Documents are written as statements

By generating a hypothetical answer, we create a text that is closer in style to the real documents. This often leads to better search results.

Example
User Question:
“How many casual leaves do employees get?”
Hypothetical Answer generated by LLM:
“Employees are entitled to 12 days of casual leave every year according to the company policy.”
Now we create an embedding of this hypothetical answer and search using it.
This usually matches better with the actual policy documents.


simple prompt

Write a short passage that answers the following question. 
The passage should look like it is taken from a company policy document.

Question: {user_question}

Passage:

(3)What is Step-back?
Step-back means taking a step back from the specific question and first asking a more general / higher-level question.
Then we use both:

The broader (step-back) question
The original specific question
Original Question (Specific)
        ↓
Generate a Broader Question (Step-back)
        ↓
Search with both questions
        ↓
Combine results

Example
Original Question:
“How many casual leaves do employees get in 2026?”
Step-back Question:
“What is the company’s leave policy?”
Now the system:

Searches for the general leave policy
Searches for the specific casual leave information
Combines both results

This gives richer and more reliable context.

Simple command:-
You are an expert at creating broader questions.

Given the original question, create a more general version of it that asks about the higher-level concept.

Original Question: {question}

Step-back Question:

(4)What is Query Decomposition?
Query Decomposition means breaking a complex question into smaller, simpler sub-questions.
Complex Question
       ↓
Break into 2–4 simpler sub-questions
       ↓
Search for each sub-question
       ↓
Combine the answers / context

This is very useful when one question contains multiple parts.

Why Do We Need It?
Complex questions are hard to retrieve in one go.
By splitting them, each part can be searched more accurately.

Example 1
Original Question:
“Compare casual leave and sick leave policy and tell which one is more flexible.”
Decomposed Sub-questions:

What is the casual leave policy?
What is the sick leave policy?
What are the differences between casual leave and sick leave?

sample prompt:
You are good at breaking complex questions into simpler sub-questions.

Break the following question into 2 to 4 simpler independent questions.
Return only the sub-questions.

Question: {user_question}

Sub-questions:

What is Contextual Compression?

Contextual Compression means reducing the retrieved documents before sending them to the LLM, by keeping only the most relevant parts.

Retrieved Chunks (can be long / noisy)
        ↓
Contextual Compression
        ↓
Only the most relevant parts remain
        ↓
Send compressed context to LLM

It removes irrelevant information and keeps only what is useful for answering the question.

How Contextual Compression Works

Retrieve top chunks using vector / hybrid search
Take each chunk + the user question
Extract or keep only the relevant parts from each chunk
Send the cleaned/compressed version to the LLM

User Question:
“How many casual leaves do employees get?”
Original Retrieved Chunk (long):
The company offers various types of leaves. Sick leave is 10 days per year. Casual leave is 12 days per year. Maternity leave is 26 weeks. Work from home is allowed 8 days per month. Employees must apply through the HR portal...
After Contextual Compression:
Casual leave is 12 days per year.
Only the relevant sentence is kept.

Main Ways to Do Contextual Compression

Method                         How it works,Complexity
LLM-based Compression        Ask LLM to extract only relevant sentences,High
Embedding-based Filtering    Keep only sentences most similar to the question,Medium
Reranker + Truncation        Rerank and keep only top parts,Medium
Keyword / Score Filtering    Remove low relevance sections,Low


prompt:-
Given the question and the document, extract only the parts that are relevant to the question.
Remove all irrelevant information.

Question: {question}

Document: {document}

Relevant Content:

Disadvantages

Extra processing step (increases latency)
LLM-based compression adds cost
Risk of removing useful information if not done carefully

What is Parent-Document Retrieval?

Parent-Document Retrieval (also called Hierarchical Retrieval) is a technique where we search using small chunks, but return the larger parent document (or bigger chunk) to the LLM.

Small chunks  →  Used for accurate search
Big parent    →  Sent to the LLM for better context

How It Works

Split documents into small child chunks (for searching)
Keep the larger parent chunks (for context)
Store relationship between child and parent
During search:
Search using small child chunks
Find the most relevant child chunks
Return their parent documents instead

Send the parent documents to the LLM
Example
Parent Document:
Employees are entitled to different types of leaves. Casual leave is 12 days per year. Sick leave is 10 days. Maternity leave is 26 weeks. All leaves must be applied through the HR portal.
Child Chunks:

Child 1: “Casual leave is 12 days per year.”
Child 2: “Sick leave is 10 days.”
Child 3: “Maternity leave is 26 weeks.”

If the user asks about casual leave:

System finds Child 1
But sends the full Parent Document to the LLM

This gives the LLM more complete context.

What is Citation?

Citation means clearly showing where the information in the answer came from.
Example:
Employees are entitled to 12 days of casual leave every year.
Source: HR_Policy.pdf, Page 4

Why Citations Are Important

Builds user trust
Makes answers verifiable
Helps in debugging wrong answers
Required in enterprise, legal, medical, and HR systems
Reduces the feeling of “black box” answers

Example:-
Employees get 12 days of casual leave.
Source: HR_Policy.pdf

How to Implement Citations
Step 1: Store Metadata
While saving chunks, always store:

source
page_number
document_id
chunk_index

What is Grounding?

Grounding means making sure the LLM’s answer is based only on the retrieved documents, not on its internal knowledge or imagination.
textRetrieved Context  →  LLM  →  Answer (must stay faithful to context)
If the answer stays true to the given context → It is well grounded.
If the LLM makes up information → It is not grounded (hallucination).

Main Grounding Techniques
1. Strong System Prompt (Most Important)
You must answer only using the provided context.
If the answer is not present in the context, say "I don't know based on the available information."
Do not use any external knowledge or make assumptions.

2. “I Don’t Know” Rule
Always instruct the model to admit when information is missing.
This is one of the strongest grounding techniques.

What are Evaluation Frameworks?

Evaluation Frameworks are tools and methods used to measure how good a RAG system is.
They help you answer:

Is the retrieval good?
Is the answer faithful to the context?
Is the answer relevant to the question?
Is the overall system improving or getting worse?

Main Things We Evaluate in RAG

Component,    What we measure                              Common Metrics
Retrieval    Are the right documents retrieved?          "Context Relevance, Recall, MRR, nDCG"
Generation   Is the answer good?,"Answer Relevance        Correctness"
Faithfulness Is the answer grounded in the context?      "Faithfulness, Hallucination Rate"
End-to-End   Overall quality,"Overall Score               User Satisfaction"

Without evaluation, you cannot improve a RAG system systematically.

Simple evaluation:-
1. Create a test set (Questions + Expected Answers)
2. Run questions through your RAG system
3. Collect: Question, Retrieved Context, Generated Answer
4. Use evaluation framework (RAGAS / TruLens / etc.)
5. Analyze scores and improve weak areas

Framework,          Type,        Strengths,                     Best For
RAGAS,            Open Source   "Most popular,                easy to use",Most RAG projects
TruLens          Open Source    Strong tracking + evaluation  Production monitoring

Winner for most production RAG evaluation:
→ RAGAS is currently the most widely used framework.


What is Synthetic Evaluation Data Generation?

It means automatically creating test questions and answers using an LLM, instead of writing them manually.

Your Documents
      ↓
LLM generates Questions + Answers
      ↓
Synthetic Evaluation Dataset
      ↓
Use it to evaluate your RAG system

This is very useful when you don’t have a ready-made test dataset.

How Synthetic Data is Generated

Common Process:

Take a document (or chunk)
Ask an LLM to generate questions from it
Ask the LLM to generate the correct answer
(Optional) Generate different types of questions
Save them as a test dataset

Sample Prompt for Generating Synthetic Data

Based on the following document, generate 3 high-quality question-answer pairs that can be used to evaluate a RAG system.

Document:
{document}

For each pair, provide:
- Question
- Ground Truth Answer

Make the questions diverse (factual, reasoning, etc.).